# Project 3 - Thread That Remember (And Forget)

#### Objective: Prove you can control short-term memory: conversations persist per thread, stay isolated between threads, and survive a restart.

Documentation - https://docs.langchain.com/oss/python/langchain/short-term-memory

##### Test
```I want to make a mushroom risotto for one person. How many calories is it, and is it vegetarian?```

##### Expected Output
```
MealAnalysis(
dish='mushroom risotto',
ingredients=[Ingredient(name='arborio rice', grams=80.0), Ingredient(name='mushrooms', grams=150.0)
total_calories=610,
dietary_flags=['vegetarian', 'gluten_free'],
confidence='high',
)
```

#### Step 1: Import Library and Environment

In [1]:
# Import Libraries
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from dotenv import load_dotenv
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.checkpoint.sqlite import SqliteSaver

# Load Environmental Files
load_dotenv(override = True)

True

#### Step 2: Create Thread Config

In [4]:
thread_config_A = {"configurable": {"thread_id": "1"}}
thread_config_B = {"configurable": {"thread_id": "2"}}

#### Step 3: Create An Agent

In [5]:
# Create Agent
agent = create_agent(model = 'gpt-5-nano', checkpointer = InMemorySaver())

#### Step 4: Call Agent With Different Threads. Thread A - Gets Order Number, Thread B - Asked About Order Number, Thread A - Asked About Order Number

In [6]:
response = agent.invoke({'messages': [HumanMessage('My order number is 190.')]}, config = thread_config_A)
print(response['messages'][-1].content)

Thanks for the order number. I don’t have access to your order details from here, but I can help you with the next steps. What would you like to do with order 190?

Options:
- Check status and tracking
- Update shipping address
- Cancel the order
- Initiate a return or refund
- Report a problem (missing/damaged item)

Please tell me the store/platform (e.g., Amazon, Walmart, a specific retailer) and what you want to do. If you’d like, I can also draft a quick message you can use to contact customer support.


In [7]:
response = agent.invoke({'messages': [HumanMessage('What is my order number?')]}, config = thread_config_B)
print(response['messages'][-1].content)

I don’t have access to your orders here, so I can’t see your order number. I can help you find it, though. Here are quick ways:

- Check your email: Look for the order confirmation from us. The order number is usually at the top and labeled “Order #” or “Order number.”
- Sign in and check: If you have an account, log in and go to your Orders or Order History to see the order numbers.
- Guest checkout: If you ordered as a guest, search your inbox for any receipts or confirmations from us and use the most recent one.

If you’d like me to help locate it here, please share:
- The email used for the order (or the last name on the order),
- The approximate date of the order,
- Any item names or the order total.

With those, I can guide you to the right place or help draft a message to support to retrieve the number.


In [8]:
response = agent.invoke({'messages': [HumanMessage('What is my order number?')]}, config = thread_config_A)
print(response['messages'][-1].content)

Based on what you told me earlier, your order number is 190.

I don’t have access to the retailer’s system to pull details. If you want help with a specific action (status, address update, cancel, return, etc.), tell me the store/platform and what you’d like to do. I can also draft a quick message to customer support for you.

If you need to locate it yourself:
- Check your confirmation email or SMS from the retailer
- Check your account’s Orders/Purchase history on their website or app
- Look at your payment receipt or bank statement for the order reference number


#### Step 5: Examine Agent State In Config.

In [9]:
snapshot = agent.get_state(thread_config_A)
print(snapshot.values.keys())
print((len(snapshot.values['messages'])))

dict_keys(['messages'])
4


#### Step 6: Test Persistant Checkpointer After Stopping Process

In [10]:
with SqliteSaver.from_conn_string('checkpoints.sqlite') as checkpointer:
    checkpointer.setup()
    agent = create_agent("gpt-5-nano", checkpointer=checkpointer)
    config = {"configurable": {"thread_id": "1"}}
    response = agent.invoke({"messages": [HumanMessage("My order number is 190.")]}, config=config)
    response = agent.invoke({"messages": [HumanMessage("What is my order number?")]}, config=config)
    print(response['messages'][-1].content)

Your order number is 190.

If you need to verify or find it again:
- Check your confirmation email from the retailer
- Sign in to your account > Orders > look for order 190
- In the mobile app, go to Orders/Purchases and locate 190

If you want, I can help draft a message to support or walk you through tracking, cancellations, or returns for this order.


In [2]:
# Restart The Kernel Before Executing This code. 
with SqliteSaver.from_conn_string('checkpoints.sqlite') as checkpointer:
    checkpointer.setup()
    agent = create_agent("gpt-5-nano", checkpointer=checkpointer)
    config = {"configurable": {"thread_id": "1"}}
    response = agent.invoke({"messages": [HumanMessage("What is my order number?")]}, config=config)
    print(response['messages'][-1].content)
    print(f"Thread has {len(agent.get_state(config).values['messages'])} messages")

Your order number is 190.

If you need to verify it:
- Check the confirmation email from the retailer
- Sign in to your account > Orders > look for order 190
- In the mobile app, go to Orders/Purchases and locate 190

How would you like to proceed with this order? track status, cancel/modify, start a return, or file a delivery issue? I can also draft a message to support if you’d like.
Thread has 6 messages
